# Gradient Boosting — Iteration by Iteration
We build it **from scratch** on a tiny 1D dataset first, then replay the same idea on real house prices.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor

plt.rcParams['figure.dpi'] = 110
np.random.seed(42)

---
## Part 1 — Build gradient boosting by hand on 1D data

We have one feature `x` and a target `y` with a curve in it.  
A single tree can't capture it well. Let's see what boosting does.

In [ ]:
# --- toy dataset ---
x = np.linspace(0, 10, 80)
y = np.sin(x) * 3 + 0.5 * x + np.random.normal(0, 0.4, len(x))

X = x.reshape(-1, 1)  # sklearn needs 2D

plt.figure(figsize=(9, 3))
plt.scatter(x, y, s=20, color='steelblue', label='data')
plt.title('Toy dataset — one feature, curvy target')
plt.xlabel('x'); plt.ylabel('y')
plt.tight_layout(); plt.show()

### Manual boosting loop

The algorithm in plain English:

```
prediction = mean(y)              # step 0: dumbest possible baseline

for each round t:
    residuals = y - prediction    # what we got WRONG so far
    tree_t    = fit a small tree on (X, residuals)
    prediction = prediction + learning_rate * tree_t.predict(X)
```

That's it. We're not fitting `y` directly after round 0.  
Every tree is a **correction machine** aimed at the current errors.

In [ ]:
LEARNING_RATE = 0.5
N_ROUNDS      = 6
MAX_DEPTH     = 2      # shallow stump-like trees — intentionally weak

# --- step 0 ---
prediction = np.full(len(y), y.mean())

trees      = []
snapshots  = []   # save (round, residuals, prediction, tree) for plotting

snapshots.append({
    'round':      0,
    'residuals':  y - prediction,
    'prediction': prediction.copy(),
    'rmse':       np.sqrt(mean_squared_error(y, prediction)),
    'tree':       None,
})

for t in range(1, N_ROUNDS + 1):
    residuals = y - prediction

    tree = DecisionTreeRegressor(max_depth=MAX_DEPTH)
    tree.fit(X, residuals)              # fit the RESIDUALS, not y
    correction = tree.predict(X)

    prediction = prediction + LEARNING_RATE * correction
    trees.append(tree)

    snapshots.append({
        'round':      t,
        'residuals':  y - prediction,  # residuals AFTER this tree
        'prediction': prediction.copy(),
        'rmse':       np.sqrt(mean_squared_error(y, prediction)),
        'correction': correction,
        'tree':       tree,
    })

print('RMSE by round:')
for s in snapshots:
    bar = '█' * int(s['rmse'] / 0.05)
    print(f"  round {s['round']:>2d}  RMSE={s['rmse']:.3f}  {bar}")

### Visualise each round

In [ ]:
n_plots = N_ROUNDS + 1
fig, axes = plt.subplots(n_plots, 2, figsize=(14, 3.5 * n_plots))

for i, s in enumerate(snapshots):
    ax_fit = axes[i, 0]
    ax_res = axes[i, 1]

    # left — data + current cumulative prediction
    ax_fit.scatter(x, y, s=18, color='steelblue', alpha=0.6, label='data')
    ax_fit.plot(x, s['prediction'], color='red', lw=2,
                label=f'prediction (RMSE={s["rmse"]:.3f})')
    if i > 0 and 'correction' in s:
        ax_fit.plot(x, snapshots[i-1]['prediction'] + LEARNING_RATE * s['correction'],
                    color='orange', lw=1.2, linestyle='--',
                    alpha=0.7, label=f'this tree × {LEARNING_RATE}')
    ax_fit.set_title(f'Round {s["round"]} — cumulative prediction')
    ax_fit.legend(fontsize=8)
    ax_fit.set_ylim(y.min() - 1, y.max() + 1)

    # right — residuals this round is trying to fit
    ax_res.bar(x, s['residuals'], width=0.1, color='coral', alpha=0.8)
    ax_res.axhline(0, color='black', lw=1)
    ax_res.set_title(f'Round {s["round"]} — residuals after prediction'
                     f'  (std={s["residuals"].std():.3f})')
    ax_res.set_ylim(-4, 4)

plt.tight_layout()
plt.show()

### Residual standard deviation shrinks every round

In [ ]:
rounds = [s['round'] for s in snapshots]
rmses  = [s['rmse']  for s in snapshots]
stds   = [s['residuals'].std() for s in snapshots]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(rounds, rmses, marker='o', color='red', lw=2)
axes[0].set_title('RMSE per round')
axes[0].set_xlabel('Boosting round'); axes[0].set_ylabel('RMSE')
for r, v in zip(rounds, rmses):
    axes[0].annotate(f'{v:.3f}', (r, v), textcoords='offset points',
                     xytext=(0, 8), ha='center', fontsize=9)

axes[1].plot(rounds, stds, marker='s', color='coral', lw=2)
axes[1].set_title('Residual std per round')
axes[1].set_xlabel('Boosting round'); axes[1].set_ylabel('std(residuals)')

plt.tight_layout()
plt.show()

print("""
Round 0: prediction = mean(y). Residuals are huge — the model knows nothing.
Round 1: first tree fits the big structure. RMSE drops sharply.
Round 2-3: second and third trees clean up mid-range errors.
Round 4-6: diminishing returns — residuals are mostly noise now.

This is why learning_rate exists: if each tree overcorrects (lr=1.0),
the next tree has nothing useful left to learn. Small lr = smaller
steps = more trees needed = better generalisation.
""")

### What does each individual tree actually look like?

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 7))

for i, (ax, snap) in enumerate(zip(axes.flat, snapshots[1:])):
    tree   = snap['tree']
    res_in = snapshots[i]['residuals']   # residuals the tree was trained on

    ax.scatter(x, res_in, s=14, color='coral', alpha=0.7,
               label='residuals fed in')
    ax.plot(x, tree.predict(X), color='navy', lw=2,
            label=f'tree prediction (max_depth={MAX_DEPTH})')
    ax.axhline(0, color='black', lw=0.8, linestyle='--')
    ax.set_title(f'Tree {i+1} — fitting round {i} residuals')
    ax.legend(fontsize=8)
    ax.set_ylim(-4, 4)

plt.tight_layout()
plt.show()

print("""
Each tree is a shallow step-function (max_depth=2 means at most 4 leaf
values). It can't fit the residuals perfectly — it only grabs the
biggest chunk. That leftover becomes the next tree's job.

Notice: by tree 4-6 the residuals are already small and noisy;
the tree can barely find a useful split. This is where early-stopping
in XGBoost would kick in.
""")

---
## Part 2 — Effect of learning rate

Same data, same number of trees (10). Only `learning_rate` changes.

In [ ]:
def manual_boost(X, y, n_rounds, lr, max_depth=2):
    pred = np.full(len(y), y.mean())
    rmse_history = [np.sqrt(mean_squared_error(y, pred))]
    for _ in range(n_rounds):
        residuals = y - pred
        tree = DecisionTreeRegressor(max_depth=max_depth)
        tree.fit(X, residuals)
        pred += lr * tree.predict(X)
        rmse_history.append(np.sqrt(mean_squared_error(y, pred)))
    return pred, rmse_history

learning_rates = [1.0, 0.5, 0.2, 0.05]
N = 30

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['red', 'orange', 'seagreen', 'steelblue']
for lr, color in zip(learning_rates, colors):
    pred, history = manual_boost(X, y, N, lr)
    axes[0].plot(history, lw=2, color=color, label=f'lr={lr}')
    axes[1].plot(x, pred, lw=2, color=color, label=f'lr={lr}  final RMSE={history[-1]:.3f}')

axes[0].set_title('RMSE over 30 rounds by learning rate')
axes[0].set_xlabel('Round'); axes[0].set_ylabel('RMSE')
axes[0].legend()

axes[1].scatter(x, y, s=15, color='black', alpha=0.4, zorder=5, label='data')
axes[1].set_title('Final prediction shape by learning rate')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print("""
lr=1.0  — aggressive. First few rounds look great but the prediction
       oscillates and stops improving early. The corrections overshoot.

lr=0.5  — reasonable for 30 trees. Good RMSE, smooth final curve.

lr=0.2  — needs more trees to converge but gets there cleanly.

lr=0.05 — barely moves after 30 rounds. With 300+ trees it would
       win — this is the XGBoost default strategy: slow + many trees.

Rule of thumb: halving lr and doubling n_estimators gives similar
accuracy but better generalisation.
""")

---
## Part 3 — Same thing on real house prices

One feature only: `GrLivArea → log(SalePrice)`.  
Watch the cumulative prediction become a better and better curve.

In [ ]:
df    = pd.read_csv('/home/ubuntu/Desktop/train.csv')
area  = df['GrLivArea'].values.reshape(-1, 1)
price = np.log1p(df['SalePrice'].values)

# sort by area for clean line plots
order = np.argsort(area.ravel())
area_s  = area[order]
price_s = price[order]

N_ROUNDS_REAL = 8
LR_REAL       = 0.5

pred_r = np.full(len(price_s), price_s.mean())
snapshots_r = [{'round': 0, 'pred': pred_r.copy(),
                'rmse': np.sqrt(mean_squared_error(price_s, pred_r)),
                'residuals': price_s - pred_r}]

for t in range(1, N_ROUNDS_REAL + 1):
    residuals = price_s - pred_r
    tree = DecisionTreeRegressor(max_depth=3)
    tree.fit(area_s, residuals)
    pred_r = pred_r + LR_REAL * tree.predict(area_s)
    snapshots_r.append({
        'round':     t,
        'pred':      pred_r.copy(),
        'rmse':      np.sqrt(mean_squared_error(price_s, pred_r)),
        'residuals': price_s - pred_r,
    })

print('RMSE by round (house prices, 1 feature):')
for s in snapshots_r:
    bar = '█' * int(s['rmse'] / 0.01)
    print(f"  round {s['round']:>2d}  RMSE={s['rmse']:.4f}  {bar}")

In [ ]:
# show rounds 0, 1, 2, 4, 8
show = [0, 1, 2, 4, 8]
fig, axes = plt.subplots(1, len(show), figsize=(18, 4), sharey=True)

for ax, idx in zip(axes, show):
    s = snapshots_r[idx]
    ax.scatter(area_s, price_s, s=6, color='steelblue', alpha=0.3)
    ax.plot(area_s, s['pred'], color='red', lw=2)
    ax.set_title(f'Round {idx}\nRMSE={s["rmse"]:.4f}')
    ax.set_xlabel('GrLivArea')

axes[0].set_ylabel('log(SalePrice)')
plt.suptitle('Boosting progression: 1 feature (GrLivArea) → log(SalePrice)', y=1.02)
plt.tight_layout()
plt.show()

print("""
Round 0: flat line at the mean — knows nothing about area.
Round 1: first tree finds the main split: large houses cost more.
         The step-function shape is typical of a depth-3 tree (8 leaf values).
Round 2: second tree smooths the kink around 1500 sqft.
Round 4: the curve is starting to look like a real trend line.
Round 8: a reasonable piece-wise linear approximation of the
         actual log(price) ~ log(area) relationship.

A single deep tree would try to memorise every point.
Boosting many shallow trees builds the same curve incrementally
without memorising — that's why it generalises.
""")

### Residuals shrinking — visually

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 10))

for ax, s in zip(axes.flat, snapshots_r):
    ax.scatter(area_s, s['residuals'], s=6, alpha=0.4, color='coral')
    ax.axhline(0, color='black', lw=1)
    ax.set_title(f'Round {s["round"]}  std={s["residuals"].std():.4f}')
    ax.set_xlabel('GrLivArea')
    ax.set_ylabel('residual')
    ax.set_ylim(-1.2, 1.2)

plt.tight_layout()
plt.show()

print("""
Each panel shows what the NEXT tree will be trained on.
The cloud compresses toward zero each round.
After round 8 the residuals look like random noise — no obvious
pattern left for a tree to exploit. That's the stopping signal.
""")

---
## Part 4 — XGBoost vs our manual boosting

XGBoost is the same algorithm plus:
- **Second-order gradients** (uses curvature of the loss, not just slope → smarter splits)
- **L1/L2 regularisation** on leaf weights
- **Approximate histogram split-finding** (fast on large data)
- **Built-in early stopping** (stops when CV score stops improving)

Let's compare them on the same single-feature problem.

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(area, price, test_size=0.2, random_state=42)

xgb = XGBRegressor(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,
    eval_metric='rmse',
    early_stopping_rounds=20,
    verbosity=0,
    random_state=42,
)
xgb.fit(X_tr, y_tr, eval_set=[(X_tr, y_tr), (X_val, y_val)], verbose=False)

evals      = xgb.evals_result()
train_rmse = evals['validation_0']['rmse']
val_rmse   = evals['validation_1']['rmse']

_, manual_history = manual_boost(X_tr, y_tr, n_rounds=len(train_rmse), lr=0.1, max_depth=3)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_rmse, label='XGBoost train', color='steelblue', lw=2)
axes[0].plot(val_rmse,   label='XGBoost val',   color='orange',    lw=2)
axes[0].plot(manual_history[:len(train_rmse)], label='Manual boost train',
             color='gray', lw=1.5, linestyle='--')
axes[0].axvline(xgb.best_iteration, color='red', linestyle=':', lw=1.5,
                label=f'early stop @ {xgb.best_iteration}')
axes[0].set_title('Train vs Val RMSE — XGBoost with early stopping')
axes[0].set_xlabel('Round'); axes[0].set_ylabel('RMSE')
axes[0].legend()

area_sorted = np.sort(area, axis=0)
axes[1].scatter(area, price, s=6, alpha=0.3, color='steelblue', label='data')
axes[1].plot(area_sorted, xgb.predict(area_sorted), color='red', lw=2, label='XGBoost')

manual_pred_sorted, _ = manual_boost(area_sorted, np.zeros(len(area_sorted)),
                                      n_rounds=xgb.best_iteration, lr=0.1)
# For visual only — refit manual on full data
manual_pred_full, _ = manual_boost(np.sort(area, axis=0),
                                    price[np.argsort(area.ravel())],
                                    n_rounds=xgb.best_iteration, lr=0.1)
axes[1].plot(np.sort(area, axis=0), manual_pred_full,
             color='gray', lw=1.5, linestyle='--', label='Manual boost')
axes[1].set_title('Fitted curves comparison')
axes[1].set_xlabel('GrLivArea'); axes[1].set_ylabel('log(SalePrice)')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'XGBoost best round: {xgb.best_iteration}')
print(f'XGBoost val RMSE:   {min(val_rmse):.4f}')
print(f'Manual boost RMSE:  {manual_history[xgb.best_iteration]:.4f}  (train only, no val)')
print("""
The early stopping chart shows the classic overfitting picture:
train RMSE keeps falling forever, but val RMSE bottoms out and then
climbs. XGBoost stops at the valley automatically.

Manual boosting has no regularisation so it fits the training data
more aggressively — lower train RMSE but would be worse on val.
""")